In [4]:
!pip install torch torchvision

import os
import torch
torch.cuda.empty_cache()

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms,models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader
import os

In [6]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
data_transforms ={
    "Training":transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.0))
        ], p=0.3),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    "Testing":
    transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
}

In [8]:
dir = "Dataset/BT-MRI Dataset/BT-MRI Dataset"

In [9]:
image_data = {
    x: datasets.ImageFolder(os.path.join(dir, x), data_transforms[x])
    for x in ["Training", "Testing"]
}

In [10]:
print("Successfully loaded datasets!")
print(f"Training images: {len(image_data['Training'])}")
print(f"Testing images: {len(image_data['Testing'])}")

Successfully loaded datasets!
Training images: 5712
Testing images: 1311


In [11]:
dataloaders = {
    x: DataLoader(image_data[x], batch_size=16, shuffle=True, num_workers=4)
    for x in ["Training", "Testing"]
}

In [15]:
model = models.resnet50(weights=ResNet50_Weights.DEFAULT)

In [16]:
for params in model.parameters():
    params.requires_grad=False

In [18]:
num_classes = 4
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(device)

In [20]:
loss_ = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(),lr=0.001)
epochs = 5

In [22]:
for param in model.fc.parameters():
    param.requires_grad = True

# Check what device PyTorch thinks it's using
print(f"Current Device: {device}")

# Check where the model actually lives
print(f"Model is on: {next(model.parameters()).device}")

for i in range(epochs+1):
    model.train()
    running_loss=0.0
    for image,label in dataloaders["Training"]:
        image,label = image.to(device),label.to(device)
        optimizer.zero_grad()

        output = model(image)
        loss = loss_(output,label)
        loss.backward()
        optimizer.step()

        running_loss +=loss.item()*image.size(0)
    epoch_loss = running_loss/len(image_data["Training"])
    print(f"Epoch{i+1}/{epochs} - Loss {epoch_loss:.4f}")

Current Device: cuda
Model is on: cuda:0
Epoch1/5 - Loss 0.9268
Epoch2/5 - Loss 0.7241
Epoch3/5 - Loss 0.6572
Epoch4/5 - Loss 0.6396
Epoch5/5 - Loss 0.6170
Epoch6/5 - Loss 0.5841


In [14]:
for i in model.parameters():
    i.requires_grad=True

In [23]:
optimizer_ft = optim.Adam(model.fc.parameters(),lr=1e-4)
epochs_ft = 30
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, mode='min', factor=0.1, patience=3)

In [24]:
print(f"Current Device: {device}")
print(f"Model is on: {next(model.parameters()).device}")
for epoch in range(epochs_ft):
    model.train()
    running_loss = 0.0
    
    for image, label in dataloaders["Training"]:
        image, label = image.to(device), label.to(device)
        
        optimizer_ft.zero_grad()
        outputs = model(image)
        loss = loss_(outputs, label)
        loss.backward()
        optimizer_ft.step()
        
        running_loss += loss.item() * image.size(0)
        
    epoch_loss = running_loss / len(image_data["Training"])
    print(f'Fine-Tune Epoch {epoch+1}/{epochs_ft} - Loss: {epoch_loss:.4f}')

Current Device: cuda
Model is on: cuda:0
Fine-Tune Epoch 1/30 - Loss: 0.5716
Fine-Tune Epoch 2/30 - Loss: 0.5746
Fine-Tune Epoch 3/30 - Loss: 0.5751
Fine-Tune Epoch 4/30 - Loss: 0.5573
Fine-Tune Epoch 5/30 - Loss: 0.5689
Fine-Tune Epoch 6/30 - Loss: 0.5730
Fine-Tune Epoch 7/30 - Loss: 0.5553
Fine-Tune Epoch 8/30 - Loss: 0.5559
Fine-Tune Epoch 9/30 - Loss: 0.5583
Fine-Tune Epoch 10/30 - Loss: 0.5579
Fine-Tune Epoch 11/30 - Loss: 0.5530
Fine-Tune Epoch 12/30 - Loss: 0.5665
Fine-Tune Epoch 13/30 - Loss: 0.5682
Fine-Tune Epoch 14/30 - Loss: 0.5654
Fine-Tune Epoch 15/30 - Loss: 0.5579
Fine-Tune Epoch 16/30 - Loss: 0.5560
Fine-Tune Epoch 17/30 - Loss: 0.5589
Fine-Tune Epoch 18/30 - Loss: 0.5503
Fine-Tune Epoch 19/30 - Loss: 0.5481
Fine-Tune Epoch 20/30 - Loss: 0.5433
Fine-Tune Epoch 21/30 - Loss: 0.5459
Fine-Tune Epoch 22/30 - Loss: 0.5429
Fine-Tune Epoch 23/30 - Loss: 0.5374
Fine-Tune Epoch 24/30 - Loss: 0.5507
Fine-Tune Epoch 25/30 - Loss: 0.5460
Fine-Tune Epoch 26/30 - Loss: 0.5332
Fine-T

In [25]:
model.eval()
correct = 0
total = 0
print("Evaluating on the standard Testing dataset...")
with torch.no_grad():
    for images, labels in dataloaders["Testing"]:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on clean Testing data: {accuracy:.2f}%')

Evaluating on the standard Testing dataset...
Accuracy on clean Testing data: 81.01%


In [26]:
import os
from torch.utils.data import DataLoader

challenging_base_dir = "Dataset/Challenging Datasets/Challenging Datasets"

challenging_folders = [
    "Blurred Dataset", 
    "Noisy Dataset", 
    "Patient Motion Artifact Dataset"
]

print("--- Starting Stress Test on Challenging Data ---")

for folder_name in challenging_folders:
    folder_path = os.path.join(challenging_base_dir, folder_name)
    test_dataset = datasets.ImageFolder(folder_path, data_transforms['Testing'])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    print(f"Accuracy on {folder_name}: {accuracy:.2f}%")

--- Starting Stress Test on Challenging Data ---
Accuracy on Blurred Dataset: 79.01%
Accuracy on Noisy Dataset: 27.13%
Accuracy on Patient Motion Artifact Dataset: 25.76%
